# Word2Vec & Dense Embeddings: Practical Implementation

Word2Vec maps discrete words into a **dense continuous vector space** $\mathbb{R}^d$ where geometric proximity reflects semantic meaning. Unlike sparse representations (One-Hot, BoW, TF-IDF), Word2Vec embeddings are compact, non-zero, and support vector linear algebra.

### Core Sections:
1. Loading Pre-trained Models via Gensim (`word2vec-google-news-300`).
2. Inspecting Vectors & Latent Dimensions.
3. Finding Nearest Neighbors & Synonyms via Cosine Similarity.
4. Vector Space Arithmetic: $\mathbf{v}_{\text{King}} - \mathbf{v}_{\text{Man}} + \mathbf{v}_{\text{Woman}} \approx \mathbf{v}_{\text{Queen}}$.
5. Semantic Outlier Detection (`doesnt_match`).
6. Training Custom Word2Vec Models from Scratch (CBOW vs Skip-Gram).
7. Average Word2Vec Implementation for Document-Level Classification.

In [ ]:
import numpy as np
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec

## 1. Exploring Pre-Trained Word2Vec (Google News 300)

Google trained this model on a 100-billion-token news corpus using Skip-Gram with Negative Sampling. It contains 300-dimensional vectors for 3 million unique words and phrases.

In [ ]:
# Load pre-trained Google News model
# Note: Model size is ~1.6 GB; once downloaded, it is cached locally by Gensim
wv = api.load('word2vec-google-news-300')

## 2. Inspecting Vectors & Dimensionality

In [ ]:
vec_king = wv['king']
print(f"Dimensionality shape of 'king': {vec_king.shape}")
print(f"First 5 float values: {vec_king[:5]}")
print(f"Vector L2 Norm (Magnitude): {np.linalg.norm(vec_king):.4f}")

## 3. Finding Nearest Neighbors & Synonyms via Cosine Similarity

The `.most_similar()` method ranks words by descending cosine similarity against the query vector.

In [ ]:
print("=== Top 5 Similar to 'cricket' ===")
for word, score in wv.most_similar('cricket', topn=5):
    print(f"  {word:15s}: {score:.4f}")

print("\n=== Top 5 Similar to 'happy' ===")
for word, score in wv.most_similar('happy', topn=5):
    print(f"  {word:15s}: {score:.4f}")

# Pairwise similarity comparison
sim_happy_glad = wv.similarity('happy', 'glad')
sim_happy_sad = wv.similarity('happy', 'sad')
print(f"\nPairwise Similarity('happy', 'glad'): {sim_happy_glad:.4f}")
print(f"Pairwise Similarity('happy', 'sad') : {sim_happy_sad:.4f}")

## 4. Vector Space Arithmetic: Solving Analogies

Because semantic concepts correspond to directional vectors, we can compute linear translations:
$$\mathbf{v}_{\text{King}} - \mathbf{v}_{\text{Man}} + \mathbf{v}_{\text{Woman}} \approx \mathbf{v}_{\text{Queen}}$$

In [ ]:
# Analogy: King - Man + Woman = ?
analogy_queen = wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
print("=== Result of (King - Man + Woman) ===")
for word, score in analogy_queen:
    print(f"  {word:15s}: {score:.4f}")

# Analogy: Paris - France + Italy = Rome
analogy_rome = wv.most_similar(positive=['Paris', 'Italy'], negative=['France'], topn=3)
print("\n=== Result of (Paris - France + Italy) ===")
for word, score in analogy_rome:
    print(f"  {word:15s}: {score:.4f}")

## 5. Semantic Outlier Detection (`doesnt_match`)

In [ ]:
words_list = ["apple", "banana", "mango", "car"]
outlier = wv.doesnt_match(words_list)
print(f"List: {words_list}")
print(f"Outlier detected: '{outlier}'")

## 6. Training Custom Word2Vec Models from Scratch

When working with domain-specific corpora (finance, biomedical, legal), pre-trained models lack proprietary jargon. You can train a custom Word2Vec model using:
- `sg=0`: Continuous Bag of Words (CBOW) — Faster, optimal for frequent words.
- `sg=1`: Skip-Gram — Slower, optimal for rare words and specialized domains.

In [ ]:
# Custom domain-specific tokenized sentences
custom_corpus = [
    ["machine", "learning", "algorithms", "optimize", "objective", "functions"],
    ["deep", "learning", "neural", "networks", "learn", "hierarchical", "representations"],
    ["natural", "language", "processing", "analyzes", "textual", "data"],
    ["transformers", "and", "attention", "mechanisms", "revolutionized", "nlp"],
    ["gradient", "descent", "minimizes", "loss", "functions", "in", "neural", "networks"]
]

# 1. Train CBOW Model (sg=0)
cbow_model = Word2Vec(
    sentences=custom_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    sg=0,
    workers=2,
    epochs=25
)
print(f"CBOW Vocabulary Size: {len(cbow_model.wv.key_to_index)}")
print(f"Vector for 'algorithms' (shape): {cbow_model.wv['algorithms'].shape}")

# 2. Train Skip-Gram Model (sg=1)
skipgram_model = Word2Vec(
    sentences=custom_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    workers=2,
    epochs=25
)
print(f"Skip-Gram Vocabulary Size: {len(skipgram_model.wv.key_to_index)}")
print(f"Vector for 'algorithms' (shape): {skipgram_model.wv['algorithms'].shape}")

## 7. Average Word2Vec: Document-Level Embeddings

To represent an entire document or sentence for machine learning classifiers (Logistic Regression, Random Forest, XGBoost), compute the arithmetic mean (centroid) of its recognized word vectors:
$$\mathbf{v}_{\text{doc}} = \frac{1}{M} \sum_{i=1}^M \mathbf{v}_{w_i} \in \mathbb{R}^d$$

In [ ]:
def compute_document_embedding(tokens, model_wv, vector_dim=50):
    """Computes the centroid vector (Average Word2Vec) for a list of tokens."""
    valid_vectors = [model_wv[word] for word in tokens if word in model_wv]
    if not valid_vectors:
        return np.zeros(vector_dim)
    return np.mean(valid_vectors, axis=0)

# Test document embedding
doc1 = ["deep", "learning", "neural", "networks"]
doc2 = ["gradient", "descent", "loss"]

v_doc1 = compute_document_embedding(doc1, cbow_model.wv, vector_dim=50)
v_doc2 = compute_document_embedding(doc2, cbow_model.wv, vector_dim=50)

print(f"Document 1 Vector Shape: {v_doc1.shape}")
print(f"Document 2 Vector Shape: {v_doc2.shape}")

# Calculate document similarity using Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
doc_sim = cosine_similarity(v_doc1.reshape(1, -1), v_doc2.reshape(1, -1))[0][0]
print(f"Cosine Similarity between Doc 1 and Doc 2: {doc_sim:.4f}")